# Suite2p — mesoscope (ScanImage multi-ROI)

Load your settings, then build the `db` with the multi-ROI geometry from the `ops.json`
written by the MATLAB helper [`mesoscope_json_from_scanimage.m`](https://github.com/MouseLand/suite2p/blob/master/helpers/mesoscope_json_from_scanimage.m).

* `settings`: **how** to process.
* `db`: **what** to process: the paths and the multi-ROI layout (`nplanes`, `nrois`, `lines`, `dx`, `dy`).

Before running, remember to set the conda/suite2p env as the python kernel.

In [11]:
import json
import logging
import suite2p
from suite2p import default_db
from suite2p.run_s2p import logger_setup, get_save_folder

## 1. Load the settings

In [12]:
settings_path = r"C:\Users\User\Documents\Code\suite2p\Python_demo_Federico\Population_mROI_GCaMP8s_settings.json"

with open(settings_path) as f:
    settings = json.load(f)

## 2. Load the multi-ROI JSON from MATLAB

In [13]:
mesoscope_ops_path = r"Z:\Lab_Share\Federico\suite2p_samples\NM023\20260209\1\ops.json"

with open(mesoscope_ops_path) as f:
    meso = json.load(f)

settings["fs"] = meso["fs"]   # frame rate of this recording

## 3. Define the db

In [14]:
db = {
    "data_path":  [
        r"Z:\Lab_Share\Federico\suite2p_samples\NM023\20260209\1",
    ],
    "save_path0": r"/path/to/results/NM023\20260209",
    "fast_disk":  "C:\\",   # local disk for the temporary binary (a raw string can't end in a backslash)
    "nplanes": meso["nplanes"],
    "nrois":   meso["nrois"],
    "lines":   meso["lines"],
    "dx":      meso["dx"],
    "dy":      meso["dy"],
}

## 4. Run

In [ ]:
db = {**default_db(), **db}
save_folder = get_save_folder(db)

# run_s2p() doesn't set up logging itself (the CLI/GUI do), so without this
# the per-stage messages are not shown. Drop any log file handler from a
# previous run of this cell so messages aren't written to an old run.log.
s2p_logger = logging.getLogger("suite2p")
for h in [h for h in s2p_logger.handlers if isinstance(h, logging.FileHandler)]:
    s2p_logger.removeHandler(h)
    h.close()
logger_setup(save_folder)   # prints to this cell + writes <save_folder>/run.log

suite2p.run_s2p(settings=settings, db=db)

2026-11-12 13:56:18,825 [INFO] 
cellpose version: 	1.0.0.1 
platform:       	win32 
python version: 	3.11.14 
torch version:  	2.10.0+cu126
2026-11-12 13:56:18,827 [INFO] data_path: ['Z:\\Lab_Share\\Federico\\suite2p_samples\\NM023\\20260209\\1']
2026-11-12 13:56:18,828 [INFO] ** Found 1 files - converting to binary **
2026-11-12 13:56:18,828 [INFO] NOTE: nplanes=1, nrois=2 => nfolders = 2


creating log file \path\to\results\NM023\20260209\suite2p\run.log


2026-11-12 13:56:33,369 [INFO] NOTE: ScanImageTiffReader not working for this tiff type, using tifffile


## 5. Open the results in the GUI

The GUI opens in its own process, so the kernel stays free and you can close the window or run this cell again without crashing anything. Set `plane` to the plane you want to look at.

In [ ]:
import subprocess
import sys
from pathlib import Path

plane = 0

# Don't call suite2p.gui.run() here: it blocks the kernel and calls sys.exit()
# when the window closes, and a second call in the same kernel crashes Qt.
# Running it in a separate Python process (same env as this kernel) avoids both.
stat_file = Path(get_save_folder({**default_db(), **db})) / f"plane{plane}" / "stat.npy"
if not stat_file.exists():
    print(f"No results at {stat_file}, opening an empty GUI")
    stat_file = ""

gui_process = subprocess.Popen([
    sys.executable, "-c",
    "import sys; from suite2p import gui; gui.run(statfile=sys.argv[1] or None)",
    str(stat_file),
])